# **DATA PULL, CLEANING AND VISUALIZATION**

## EDA

### **Data Pull**
This code pulls up confirmed exoplanet data from the NASA Exoplanet Archive's public TAP (Table Access Protocol) API and saves it as a raw CSV for the
DataPrep_EDA tab of my project.

> On githuhb repo - # fetch_exoplanet_data.py

> Also downloaded the data from -

https://phl.upr.edu/hwc/data



In [17]:
# Impoting Libraries.
import pandas as pd
import requests

In [18]:
# TAP STuff - table access protocol (like API request for astronomy)
# what I'm doing after this "requests.get(TAP_URL, params=params)"- This is an HTTP request to a server, with structured parameters,
# returning structured data back (CSV) that the code parses programmatically.
# That's precisely what an API is: a defined way for code to talk to a server and get data back, no browser, no manual download, no clicking through a webpage.
TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

In [19]:
# Columns from the Planetary Systems Composite Parameters table
# (pscomppars): one best estimate row per confirmed planet.
COLUMNS = [
    "pl_name",        # planet name
    "hostname",       # host star name
    "discoverymethod",# transit, radial velocity, imaging, microlensing, etc. etc.
    "disc_year",      # year of discovery
    "pl_orbper",      # orbital period (days)
    "pl_rade",        # planet radius (Earth radii)
    "pl_bmasse",      # planet mass (Earth masses)
    "pl_eqt",         # equilibrium temperature (K)
    "st_teff",        # host star effective temperature (K)
    "st_rad",         # host star radius (solar radii)
    "st_mass",        # host star mass (solar masses)
    "sy_dist",        # system distance from Earth (parsecs)
]


In [20]:
def fetch_raw_data() -> pd.DataFrame:
    query = f"select {','.join(COLUMNS)} from pscomppars"
    params = {"query": query, "format": "csv"}
    response = requests.get(TAP_URL, params=params, timeout=60)
    response.raise_for_status()

# requests.Response has .url with the final, fully-encoded GET URL.
# This is the exact request in the DataPrep_EDA section on my site.
    print("Request URL:", response.url)

    from io import StringIO
    return pd.read_csv(StringIO(response.text))

if __name__ == "__main__":
    df = fetch_raw_data()
    df.to_csv("exoplanets_raw.csv", index=False)
    print(f"Saved {len(df)} rows to exoplanets_raw.csv")
    print(df.head())

Request URL: https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+pl_name%2Chostname%2Cdiscoverymethod%2Cdisc_year%2Cpl_orbper%2Cpl_rade%2Cpl_bmasse%2Cpl_eqt%2Cst_teff%2Cst_rad%2Cst_mass%2Csy_dist+from+pscomppars&format=csv
Saved 6366 rows to exoplanets_raw.csv
         pl_name     hostname  discoverymethod  disc_year    pl_orbper  \
0      HD 2039 b      HD 2039  Radial Velocity       2002  1120.000000   
1      HAT-P-8 b      HAT-P-8          Transit       2008     3.076340   
2        K2-43 b        K2-43          Transit       2016     3.471149   
3  Kepler-1753 b  Kepler-1753          Transit       2021    16.004601   
4  Kepler-1176 b  Kepler-1176          Transit       2016    24.173858   

     pl_rade  pl_bmasse   pl_eqt  st_teff  st_rad  st_mass   sy_dist  
0  12.700000  1999.1507   210.84   5945.0   1.190    1.230   85.6922  
1  15.692600   406.8224  1713.00   6200.0   1.570    1.270  211.5530  
2   4.510000    18.5000   939.30   3840.6   0.542    0.571  182.5360 

### **CLEANING AND VISUALIZATION**

 > On githuhb repo - # clean_and_visualize.py

Cleans the raw NASA Exoplanet Archive pull, merges in habitability labels from the PHL Habitable Worlds Catalog and produces the 11 visualizations and two data-preview images used on the Cleaning & Prep and EDA tabs.

Inputs:

    exoplanets_raw.csv   from fetch_exoplanet_data.py

    hwc.csv              the "Full Catalog (CSV)" downloaded from phl.upr.edu/hwc/data

    hwc_table_all.csv    the 70-row shortlist table (Table 1 + Table 2) from the same page, saved as CSV

Outputs:

    exoplanets_clean.csv       cleaned + merged dataset (20 columns)

    charts/01_missingness.png ... charts/12_top_esi_table.png

    charts/raw_preview.png, charts/clean_preview.png

In [21]:
# Importing more libraries -
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [22]:
# the theme to match the site's CSS variables in style.css # Looked at documentations etc. for this one.
BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "grid.color": LINE,
    "font.size": 11, "axes.titlesize": 13, "axes.titlecolor": TEXT,})

os.makedirs("charts", exist_ok=True)

df = pd.read_csv("exoplanets_raw.csv")

In [23]:
# CLEANING
clean = df.copy()
# checking for duplicate planet names
print("Duplicate planet names:", clean["pl_name"].duplicated().sum())

# checking for negative or zero values in the numeric columns, which would be
# physically impossible for these measurements
numeric_cols = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]
for col in numeric_cols:
    n_bad = (clean[col] <= 0).sum()
    print(f"{col}: {n_bad} values <= 0")

# checking how much is missing in each column
print("\nMissing values per column:")
print(clean.isna().sum())

# 157 planets sit above the ~13 Jupiter mass (4131 Earth mass) boundary,
# where "planet" starts overlapping with "brown dwarf" - flagged rather
# than dropped, since they are still confirmed entries in the archive
clean["is_borderline_massive"] = clean["pl_bmasse"] > 4131
print("\nBorderline massive planets flagged:", clean["is_borderline_massive"].sum())

Duplicate planet names: 0
pl_orbper: 0 values <= 0
pl_rade: 0 values <= 0
pl_bmasse: 0 values <= 0
pl_eqt: 0 values <= 0
st_teff: 0 values <= 0
st_rad: 0 values <= 0
st_mass: 0 values <= 0
sy_dist: 0 values <= 0

Missing values per column:
pl_name              0
hostname             0
discoverymethod      0
disc_year            0
pl_orbper          350
pl_rade             50
pl_bmasse           31
pl_eqt             534
st_teff            304
st_rad             328
st_mass              9
sy_dist             28
dtype: int64

Borderline massive planets flagged: 157


- duplicate planet names and no negative/zero physical values were found in any numeric column on inspection, so nothing needed to be dropped on those grounds.

- Missingness is left as NaN rather than imputed here, since the right way to fill it depends on which model uses that column later.

- Flag planets above the ~13-Jupiter-mass (4131 Earth-mass)boundary, where "planet" starts overlapping with "brown dwarf" by common convention.

- Kept (they are confirmed in the archive) but flagged rather than treated as ordinary planets.
- As for the plots: JUST saving my images and charts in the back instead of plt.show() here.

In [24]:
# Discretize planet size into standard exoplanet-science size classes.
def size_class(r):
    if pd.isna(r): return np.nan
    if r < 1.25: return "Earth-sized"
    if r < 2.0: return "Super-Earth"
    if r < 6.0: return "Sub-Neptune/Neptune"
    return "Giant"
clean["size_class"] = clean["pl_rade"].apply(size_class)

# Discretize equilibrium temperature into a rough habitability zone.
# ~200-320 K is the loose range where liquid water is plausible.
def temp_zone(t):
    if pd.isna(t): return np.nan
    if t < 200: return "Too Cold"
    if t <= 320: return "Temperate"
    return "Too Hot"
clean["temp_zone"] = clean["pl_eqt"].apply(temp_zone)

In [25]:
# MERGE WITH PHL HABITABLE WORLDS CATALOG
hwc = pd.read_csv("hwc_full.csv")
hwc_cols = hwc[["P_NAME", "P_HABITABLE", "P_ESI", "P_TYPE",
                "P_HABZONE_OPT", "P_HABZONE_CON"]].rename(columns={"P_NAME": "pl_name"})

clean = clean.merge(hwc_cols, on="pl_name", how="left")
clean["hwc_habitable_label"] = clean["P_HABITABLE"].map(
    {0: "Not habitable", 1: "Conservative sample", 2: "Optimistic sample"})

clean.to_csv("exoplanets_clean.csv", index=False)
print("Cleaned + merged file saved:", clean.shape)
print(clean["size_class"].value_counts(dropna=False))
print(clean["temp_zone"].value_counts(dropna=False))
print(clean["hwc_habitable_label"].value_counts(dropna=False))

# PREVIEW TABLE IMAGES
def save_table_image(d, cols, fname, title):
    sample = d[cols].head(6)
    fig, ax = plt.subplots(figsize=(11, 2.2))
    ax.axis("off")
    ax.set_title(title, loc="left", color=GOLD, fontsize=12, pad=14)
    tbl = ax.table(cellText=sample.round(2).astype(str).values,
                    colLabels=cols, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.6)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor(LINE)
        if row == 0:
            cell.set_facecolor(PANEL)
            cell.set_text_props(color=GOLD, weight="bold")
        else:
            cell.set_facecolor(BG)
            cell.set_text_props(color=TEXT)
    plt.savefig(f"charts/{fname}", dpi=150, bbox_inches="tight")
    plt.close()

save_table_image(df, ["pl_name", "hostname", "discoverymethod", "disc_year", "pl_rade", "pl_bmasse"],
                  "raw_preview.png", "Raw data (as pulled from the API)")
save_table_image(clean, ["pl_name", "pl_rade", "size_class", "pl_eqt", "temp_zone", "is_borderline_massive"],
                  "clean_preview.png", "Cleaned data (derived columns added)")

Cleaned + merged file saved: (6366, 21)
size_class
Sub-Neptune/Neptune    2393
Giant                  2149
Super-Earth            1198
Earth-sized             576
NaN                      50
Name: count, dtype: int64
temp_zone
Too Hot      5142
NaN           534
Temperate     362
Too Cold      328
Name: count, dtype: int64
hwc_habitable_label
Not habitable          5495
NaN                     801
Optimistic sample        41
Conservative sample      29
Name: count, dtype: int64


### **VISUALIZATIONS**

Plotting and saving charts in the backend

In [26]:
# VISUALIZATIONS
# JUST saving my images and charts in the back instead of plt.show() here.
# 1. Missingness per column
miss = df.isna().mean().sort_values(ascending=False) * 100
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(miss.index, miss.values, color=TEAL)
ax.set_xlabel("% missing")
ax.set_title("Missing data by column")
ax.invert_yaxis()
plt.tight_layout(); plt.savefig("charts/01_missingness.png", dpi=150); plt.close()

# 2. Discovery method counts (log scale)
dm = df["discoverymethod"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(dm.index[::-1], dm.values[::-1], color=GOLD)
ax.set_xscale("log")
ax.set_xlabel("Number of confirmed planets (log scale)")
ax.set_title("Confirmed planets by discovery method")
plt.tight_layout(); plt.savefig("charts/02_discovery_method.png", dpi=150); plt.close()

# 3. Discoveries per year
yr = df["disc_year"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(yr.index, yr.values, color=RUST, linewidth=2)
ax.fill_between(yr.index, yr.values, color=RUST, alpha=0.15)
ax.set_xlabel("Year"); ax.set_ylabel("Planets confirmed")
ax.set_title("Confirmed exoplanet discoveries per year")
plt.tight_layout(); plt.savefig("charts/03_discoveries_per_year.png", dpi=150); plt.close()

# 4. Planet radius distribution (log)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["pl_rade"].dropna(), bins=40, color=TEAL)
ax.set_xscale("log")
ax.set_xlabel("Planet radius (Earth radii, log scale)"); ax.set_ylabel("Count")
ax.set_title("Distribution of planet radius")
plt.tight_layout(); plt.savefig("charts/04_radius_distribution.png", dpi=150); plt.close()

# 5. Planet mass distribution (log)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["pl_bmasse"].dropna(), bins=40, color=GOLD)
ax.set_xscale("log")
ax.set_xlabel("Planet mass (Earth masses, log scale)"); ax.set_ylabel("Count")
ax.set_title("Distribution of planet mass")
plt.tight_layout(); plt.savefig("charts/05_mass_distribution.png", dpi=150); plt.close()

# 6. Mass vs radius scatter
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df["pl_rade"], df["pl_bmasse"], s=8, alpha=0.4, color=TEAL)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Radius (Earth radii, log)"); ax.set_ylabel("Mass (Earth masses, log)")
ax.set_title("Planet mass vs. radius")
plt.tight_layout(); plt.savefig("charts/06_mass_vs_radius.png", dpi=150); plt.close()

# 7. Orbital period vs equilibrium temperature
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df["pl_orbper"], df["pl_eqt"], s=8, alpha=0.4, color=RUST)
ax.set_xscale("log")
ax.set_xlabel("Orbital period (days, log scale)"); ax.set_ylabel("Equilibrium temperature (K)")
ax.set_title("Orbital period vs. equilibrium temperature")
plt.tight_layout(); plt.savefig("charts/07_period_vs_temp.png", dpi=150); plt.close()

# 8. Host star temperature histogram
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["st_teff"].dropna(), bins=40, color=GOLD)
ax.axvline(5778, color=TEXT, linestyle="--", linewidth=1, label="Sun (5778 K)")
ax.set_xlabel("Host star effective temperature (K)"); ax.set_ylabel("Count")
ax.set_title("Host star temperature")
ax.legend()
plt.tight_layout(); plt.savefig("charts/08_star_temperature.png", dpi=150); plt.close()

# 9. Size class breakdown
sc = clean["size_class"].value_counts()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(sc.index, sc.values, color=[TEAL, GOLD, RUST, DIM])
ax.set_ylabel("Count")
ax.set_title("Planets by size class")
plt.tight_layout(); plt.savefig("charts/09_size_class.png", dpi=150); plt.close()

# 10. Temperature zone breakdown
tz = clean["temp_zone"].value_counts().reindex(["Too Cold", "Temperate", "Too Hot"])
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(tz.index, tz.values, color=[TEAL, GOLD, RUST])
ax.set_ylabel("Count")
ax.set_title("Planets by equilibrium-temperature zone")
plt.tight_layout(); plt.savefig("charts/10_temp_zone.png", dpi=150); plt.close()

# 11. Official HWC habitability classification
counts = clean["hwc_habitable_label"].value_counts().reindex(
    ["Not habitable", "Conservative sample", "Optimistic sample"]
)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(counts.index, counts.values, color=[DIM, TEAL, GOLD])
ax.set_ylabel("Count")
ax.set_title("Official Habitable Worlds Catalog classification")
for i, v in enumerate(counts.values):
    ax.text(i, v + 40, str(int(v)), ha="center", color=TEXT, fontsize=10)
plt.tight_layout(); plt.savefig("charts/11_hwc_habitability.png", dpi=150); plt.close()

# 12. Top 10 most Earth-like worlds by ESI, from the shortlist table
short = pd.read_csv("hwc_table_all.csv")
short.columns = [re.sub(r"<[^>]+>", "", c) for c in short.columns]
short = short.sort_values("ESI", ascending=False).head(10)
short_display = short[["Name", "Type", "ESI"]].round({"ESI": 3})

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.axis("off")
ax.set_title("Top 10 potentially habitable worlds, by Earth Similarity Index",
             loc="left", color=GOLD, fontsize=12, pad=14)
tbl = ax.table(cellText=short_display.values, colLabels=["Name", "Type", "ESI"],
                cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.5)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor(LINE)
    if row == 0:
        cell.set_facecolor(PANEL); cell.set_text_props(color=GOLD, weight="bold")
    else:
        cell.set_facecolor(BG); cell.set_text_props(color=TEXT)
plt.savefig("charts/12_top_esi_table.png", dpi=150, bbox_inches="tight")
plt.close()

print("All charts saved to charts/")

All charts saved to charts/


In [27]:
# Downloading my files!
import shutil
shutil.make_archive('charts', 'zip', 'charts')
from google.colab import files
files.download('charts.zip')
files.download('exoplanets_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>